Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public
License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later
version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied
warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see
<https://www.gnu.org/licenses/>.

# H11 v2: Delta-Rule Writing, a Pilot

*A hunch* ([docs/hypotheses/README.md § H11](../docs/hypotheses/README.md#h11-a-hebbian-hopfield-transformer)): one
transformer block whose memory is a Hopfield matrix written by Hebb's rule as the tokens arrive and read by noisy,
temperature-controlled mean-field settling, the whole trained by backpropagation. H11 v1 was refuted (its
notebook, `h11_hebbian_hopfield_transformer.ipynb`); this pilot writes the Hopfield matrix by the delta rule instead of
Hebb's, as pre-registered in the 2026-10-14 journal (*H11 v2, Delta-Rule Writing: A Pilot*) before it was run. The task,
the block and v1's classes are reused unchanged.

The task is associative recall after [Ba et al. 2016](https://arxiv.org/abs/1610.06258). This explores; nothing the
benchmark depends on lives here.

In [1]:
import math, os, time
import multiprocessing as mp
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
matplotlib.use("svg")
import matplotlib.pyplot as plt

print("torch", torch.__version__, "numpy", np.__version__, "matplotlib", matplotlib.__version__)

torch 2.14.1+cpu numpy 2.5.4 matplotlib 3.11.2


## The Task

Sixteen key-value pairs, keys drawn without repetition from 128 key tokens and values from 16 value tokens, then a
query mark and one of the sixteen keys; the answer is that key's value. Chance is 1/16.

In [2]:
NK, NV, NPAIR = 128, 16, 16          # key vocabulary, value vocabulary, pairs per sequence
QUERY = NK + NV                      # the query mark
VOCAB = NK + NV + 1
DM = 64                              # model width


def make_batch(gen, b, n=NPAIR):
    keys = torch.stack([torch.randperm(NK, generator=gen)[:n] for _ in range(b)])
    vals = torch.randint(0, NV, (b, n), generator=gen)
    pick = torch.randint(0, n, (b,), generator=gen)
    seq = torch.empty(b, 2 * n + 2, dtype=torch.long)
    seq[:, 0:2 * n:2] = keys
    seq[:, 1:2 * n:2] = NK + vals
    seq[:, -2] = QUERY
    seq[:, -1] = keys[torch.arange(b), pick]
    return seq, vals[torch.arange(b), pick]


def answer_by_reading(seq):
    """The answer found by reading the sequence, independently of how make_batch built it."""
    out = []
    for row in seq.tolist():
        q = row[-1]
        out.append(next(row[i + 1] - NK for i in range(0, len(row) - 2, 2) if row[i] == q))
    return torch.tensor(out)


s, y = make_batch(torch.Generator().manual_seed(0), 4)
print(s[0].tolist(), "->", y[0].item())

[44, 134, 94, 138, 121, 133, 13, 128, 31, 142, 116, 139, 71, 128, 84, 138, 105, 140, 14, 131, 15, 138, 22, 129, 48, 138, 11, 137, 30, 140, 77, 141, 144, 13] -> 0


## The Models

One block each: an embedding with a one-token shift (each position sees its token and the one before), a memory
mixer read at the last position, a residual, layer norm, an MLP and a readout. Only the mixer differs.

- `NoMemory`: the control, which sees only the query.
- `Softmax`: one head of softmax attention, a modern Hopfield step ([Ramsauer et al. 2020](https://arxiv.org/abs/2008.02217)).
- `Hebbian`: linear attention as a fast-weight memory `M <- lam M + g v k^T`
  ([Schlag, Irie and Schmidhuber 2021](https://arxiv.org/abs/2102.11174)), read once.
- `HebbHopfield`, the hybrid: patterns `p = tanh([k; v])` written by the same rule into a symmetric Hopfield matrix
  `W <- lam W + g p p^T / D` ([Hopfield 1982](https://doi.org/10.1073/pnas.79.8.2554)), read by `K` synchronous
  steps of `xi <- tanh(beta W xi + alpha xi0 + sigma eps)` from `xi0 = [tanh(q); 0]`: mean-field settling at inverse
  temperature `beta`, with Gaussian noise in training standing in for a Boltzmann machine's sampling
  ([Ackley, Hinton and Sejnowski 1985](https://doi.org/10.1207/s15516709cog0901_7)). It is a stand-in, not a
  Boltzmann machine: the units are continuous and nothing is sampled at test time.

Because a decaying sum of outer products is linear in its writes, both memories are computed as one weighted sum
rather than a loop over time; the first check below holds the two equal.

In [3]:
class Block(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, DM)
        self.mixer = mixer
        self.norm = nn.LayerNorm(DM)
        self.mlp = nn.Sequential(nn.Linear(DM, 4 * DM), nn.GELU(), nn.Linear(4 * DM, DM))
        self.out = nn.Linear(DM, NV)

    def forward(self, seq):
        e = self.emb(seq)
        prev = torch.cat([torch.zeros_like(e[:, :1]), e[:, :-1]], 1)
        x = torch.cat([e, prev], -1)
        h = e[:, -1] + self.mixer(x)
        h = h + self.mlp(self.norm(h))
        return self.out(h)


class NoMemory(nn.Module):
    def __init__(self):
        super().__init__()
        self.o = nn.Linear(2 * DM, DM)

    def forward(self, x):
        return self.o(x[:, -1])


class Softmax(nn.Module):
    def __init__(self, dk, dv):
        super().__init__()
        self.dk, self.dv = dk, dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.o = nn.Linear(dv, DM)

    def forward(self, x):
        q, k, v = self.q(x[:, -1]), self.k(x[:, :-1]), self.v(x[:, :-1])
        a = torch.softmax((k @ q.unsqueeze(-1)).squeeze(-1) / math.sqrt(self.dk), -1)
        return self.o((a.unsqueeze(-1) * v).sum(1))


def hebbian_sum(lam, g, a, b):
    """sum_s lam^(T-1-s) g_s a_s b_s^T: the state after the loop m <- lam m + g_s a_s b_s^T over s = 0 .. T-1."""
    t = a.shape[1]
    decay = lam ** torch.arange(t - 1, -1, -1, dtype=a.dtype)
    return torch.einsum("bs,bsi,bsj->bij", g * decay, a, b)


class Hebbian(nn.Module):
    def __init__(self, dk, dv):
        super().__init__()
        self.dk, self.dv = dk, dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.g = nn.Linear(2 * DM, 1)
        self.lam_logit = nn.Parameter(torch.tensor(4.0))
        self.o = nn.Linear(dv, DM)

    def forward(self, x):
        k = F.normalize(self.k(x[:, :-1]), dim=-1)
        v, g = self.v(x[:, :-1]), torch.sigmoid(self.g(x[:, :-1]))[..., 0]
        m = hebbian_sum(torch.sigmoid(self.lam_logit), g, v, k)
        q = F.normalize(self.q(x[:, -1]), dim=-1)
        return self.o((m @ q.unsqueeze(-1)).squeeze(-1))


class HebbHopfield(nn.Module):
    def __init__(self, dk, dv, steps=5, sigma=0.1, learn_beta=True):
        super().__init__()
        self.dk, self.dv, self.steps, self.sigma = dk, dv, steps, sigma
        self.d = dk + dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.g = nn.Linear(2 * DM, 1)
        self.lam_logit = nn.Parameter(torch.tensor(4.0))
        self.log_beta = nn.Parameter(torch.tensor(0.0), requires_grad=learn_beta)
        self.log_alpha = nn.Parameter(torch.tensor(0.0))
        self.o = nn.Linear(dv, DM)

    def settle(self, x):
        """The memory, the start state and every state of the settling, for the read and for diagnostics."""
        b = x.shape[0]
        p = torch.tanh(torch.cat([self.k(x[:, :-1]), self.v(x[:, :-1])], -1))
        g = torch.sigmoid(self.g(x[:, :-1]))[..., 0]
        w = hebbian_sum(torch.sigmoid(self.lam_logit), g, p, p) / self.d
        xi0 = torch.cat([torch.tanh(self.q(x[:, -1])), x.new_zeros(b, self.dv)], -1)
        beta, alpha = self.log_beta.exp(), self.log_alpha.exp()
        states = [xi0]
        for _ in range(self.steps):
            h = beta * (w @ states[-1].unsqueeze(-1)).squeeze(-1) + alpha * xi0
            if self.training and self.sigma > 0:
                h = h + self.sigma * torch.randn_like(h)
            states.append(torch.tanh(h))
        return w, states

    def forward(self, x):
        _, states = self.settle(x)
        return self.o(states[-1][:, self.dk:])


class Delta(nn.Module):
    """DeltaNet's fast weights: M <- lam M + g (v - M k) k^T, written by the error, read once."""

    def __init__(self, dk, dv):
        super().__init__()
        self.dk, self.dv = dk, dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.g = nn.Linear(2 * DM, 1)
        self.lam_logit = nn.Parameter(torch.tensor(4.0))
        self.o = nn.Linear(dv, DM)

    def forward(self, x):
        b, t, _ = x.shape
        k = F.normalize(self.k(x[:, :-1]), dim=-1)
        v, g = self.v(x[:, :-1]), torch.sigmoid(self.g(x[:, :-1]))
        lam = torch.sigmoid(self.lam_logit)
        m = x.new_zeros(b, self.dv, self.dk)
        for s in range(t - 1):
            err = v[:, s] - (m @ k[:, s, :, None])[..., 0]
            m = lam * m + g[:, s, :, None] * err[:, :, None] * k[:, s, None, :]
        q = F.normalize(self.q(x[:, -1]), dim=-1)
        return self.o((m @ q[..., None])[..., 0])


class HebbHopfield2(nn.Module):
    """H11 v2: heads of autoassociative Hopfield memories written by the delta (pseudo-inverse) rule
    W <- lam W + g (p - W p) p^T / |p|^2, read by settling with a learned inverse temperature per step."""

    def __init__(self, heads, dk, dv, steps=5, sigma=0.1, delta=True, anneal=True):
        super().__init__()
        self.h, self.dk, self.dv, self.steps, self.sigma, self.delta = heads, dk, dv, steps, sigma, delta
        self.d = dk + dv
        self.q, self.k, self.v = (nn.Linear(2 * DM, heads * dk), nn.Linear(2 * DM, heads * dk),
                                  nn.Linear(2 * DM, heads * dv))
        self.g = nn.Linear(2 * DM, heads)
        self.lam_logit = nn.Parameter(torch.full((heads,), 4.0))
        self.log_beta = nn.Parameter(torch.zeros(steps if anneal else 1))
        self.log_alpha = nn.Parameter(torch.zeros(heads))
        self.o = nn.Linear(heads * dv, DM)

    def settle(self, x):
        b, t, _ = x.shape
        H = self.h
        k = self.k(x[:, :-1]).view(b, t - 1, H, self.dk)
        v = self.v(x[:, :-1]).view(b, t - 1, H, self.dv)
        p = torch.tanh(torch.cat([k, v], -1))                               # b, s, H, D
        g = torch.sigmoid(self.g(x[:, :-1]))                                # b, s, H
        lam = torch.sigmoid(self.lam_logit)[None, :, None, None]
        if self.delta:
            w = x.new_zeros(b, H, self.d, self.d)
            for s in range(t - 1):
                ps = p[:, s]
                err = ps - (w @ ps[..., None])[..., 0]
                w = lam * w + (g[:, s, :, None, None] * err[..., None] * ps[..., None, :]
                               / (ps * ps).sum(-1)[..., None, None].clamp_min(1e-6))
        else:
            decay = torch.sigmoid(self.lam_logit)[None, :] ** torch.arange(t - 2, -1, -1, dtype=x.dtype)[:, None]
            w = torch.einsum("bsh,bshi,bshj->bhij", g * decay, p, p) / self.d
        xi0 = torch.cat([torch.tanh(self.q(x[:, -1]).view(b, H, self.dk)), x.new_zeros(b, H, self.dv)], -1)
        alpha = self.log_alpha.exp()[None, :, None]
        states = [xi0]
        for i in range(self.steps):
            beta = self.log_beta[min(i, len(self.log_beta) - 1)].exp()
            h = beta * (w @ states[-1][..., None])[..., 0] + alpha * xi0
            if self.training and self.sigma > 0:
                h = h + self.sigma * torch.randn_like(h)
            states.append(torch.tanh(h))
        return w, states

    def forward(self, x):
        _, states = self.settle(x)
        return self.o(states[-1][..., self.dk:].reshape(x.shape[0], -1))


MODELS = {
    "delta-wide": lambda: Delta(64, 16),
    "h11v2": lambda: HebbHopfield2(4, 8, 8),
    "h11v2-1step": lambda: HebbHopfield2(4, 8, 8, steps=1),
}

## Checks Before the Run

Each was seen failing once, deliberately, before it was trusted (the journal says how): the task's answer agrees with
an independent reading of the sequence, the vectorised Hebbian sum agrees with the loop it replaces, and the hybrid's
memory is symmetric, as a Hopfield matrix must be for its energy to exist.

In [4]:
for n in (NPAIR, 48):
    seq, y = make_batch(torch.Generator().manual_seed(7), 512, n)
    assert torch.equal(answer_by_reading(seq), y), "the task's answer disagrees with the sequence"

gen = torch.Generator().manual_seed(8)
a, b, g = torch.randn(3, 9, 5, generator=gen), torch.randn(3, 9, 4, generator=gen), torch.rand(3, 9, generator=gen)
lam = torch.tensor(0.9)
m = torch.zeros(3, 5, 4)
for s in range(9):
    m = lam * m + g[:, s, None, None] * a[:, s, :, None] * b[:, s, None, :]
assert torch.allclose(hebbian_sum(lam, g, a, b), m, atol=1e-5), "the vectorised Hebbian sum disagrees with the loop"

torch.manual_seed(0)
blk = Block(HebbHopfield(16, 16)).eval()
with torch.no_grad():
    e = blk.emb(seq[:8])
    x = torch.cat([e, torch.cat([torch.zeros_like(e[:, :1]), e[:, :-1]], 1)], -1)
    w, _ = blk.mixer.settle(x)
assert torch.allclose(w, w.transpose(1, 2)), "the Hopfield matrix is not symmetric"
print("checks passed")

checks passed


## The Run

As pre-registered: Adam, learning rate 3e-3, 1000 steps of 256 fresh sequences, seeds 0 to 2 (weights from
`torch.manual_seed(seed)`, data from generator seed `1000 + seed`), one test set of 10 000 sequences from generator
seed 999. Runs go four at a time in forked processes, one thread each; each is deterministic given its seed.

In [5]:
STEPS, BATCH, LR, TEST_N, SEEDS = 1000, 256, 3e-3, 10000, range(3)


def run(job):
    name, seed = job
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    model = Block(MODELS[name]())
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
    gen = torch.Generator().manual_seed(1000 + seed)
    t0 = time.process_time()
    for _ in range(STEPS):
        seq, y = make_batch(gen, BATCH)
        loss = F.cross_entropy(model(seq), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    cpu = time.process_time() - t0
    model.eval()
    with torch.no_grad():
        seq, y = make_batch(torch.Generator().manual_seed(999), TEST_N)
        acc = (model(seq).argmax(-1) == y).float().mean().item()
        extra = {}
        if isinstance(model.mixer, HebbHopfield2):
            e = model.emb(seq[:2000])
            x = torch.cat([e, torch.cat([torch.zeros_like(e[:, :1]), e[:, :-1]], 1)], -1)
            w, states = model.mixer.settle(x)
            extra = dict(beta=[round(b, 3) for b in model.mixer.log_beta.exp().tolist()],
                         asym=((w - w.transpose(-1, -2)).norm(dim=(-2, -1)) / w.norm(dim=(-2, -1))).mean().item(),
                         last_move=(states[-1] - states[-2]).norm(dim=-1).mean().item(),
                         first_move=(states[1] - states[0]).norm(dim=-1).mean().item())
    return name, seed, acc, cpu, math.isfinite(loss.item()), extra, model.state_dict()


jobs = [(n, s) for n in MODELS for s in SEEDS]
t0 = time.time()
with mp.get_context("fork").Pool(4) as pool:
    results = pool.map(run, jobs, chunksize=1)
print(f"{len(results)} runs in {(time.time() - t0) / 60:.1f} min of wall clock")
acc = {n: np.array([r[2] for r in results if r[0] == n]) for n in MODELS}
assert all(r[4] for r in results), "a run ended with a non-finite loss"
for n in MODELS:
    print(f"{n:14s}", " ".join(f"{a:.4f}" for a in acc[n]))

9 runs in 10.8 min of wall clock
delta-wide     0.9943 0.9972 0.9960
h11v2          0.8491 0.9522 0.8569
h11v2-1step    0.8634 0.9166 0.7940


## The Statistic

As pre-registered: per model, the mean over seeds with a 95% t-interval; per comparison, the paired difference per
seed with its 95% t-interval (t = 4.303 at 2 degrees of freedom).

In [6]:
T4 = 4.303


def interval(xs):
    m, h = xs.mean(), T4 * xs.std(ddof=1) / math.sqrt(len(xs))
    return m, m - h, m + h


print("model           mean    95% interval")
for n in MODELS:
    m, lo, hi = interval(acc[n])
    print(f"{n:14s}  {m:.4f}  [{lo:.4f}, {hi:.4f}]")
print()
print("paired difference             mean     95% interval")
for a, b in [("h11v2", "delta-wide"), ("h11v2", "h11v2-1step")]:
    m, lo, hi = interval(acc[a] - acc[b])
    print(f"{a:13s} - {b:13s}  {m:+.4f}  [{lo:+.4f}, {hi:+.4f}]")

model           mean    95% interval
delta-wide      0.9958  [0.9922, 0.9995]
h11v2           0.8861  [0.7435, 1.0287]
h11v2-1step     0.8580  [0.7053, 1.0107]

paired difference             mean     95% interval
h11v2         - delta-wide     -0.1098  [-0.2493, +0.0298]
h11v2         - h11v2-1step    +0.0281  [-0.0692, +0.1253]


## What the Settling Does

The learned inverse temperature of each settling step, the asymmetry of the delta-written matrix
(`|W - W^T| / |W|`, mean over heads and 2000 test sequences), and how far the state moves in the first and the last
step of settling: a last move much smaller than the first means the dynamics settle rather than oscillate.

In [7]:
for r in results:
    if r[5]:
        e = r[5]
        print(f"{r[0]:12s} seed {r[1]}  beta {e['beta']}  asymmetry {e['asym']:.3f}  "
              f"first move {e['first_move']:.3f}  last move {e['last_move']:.4f}")

h11v2        seed 0  beta [0.518, 0.384, 0.312, 0.305, 3.207]  asymmetry 0.214  first move 0.458  last move 1.3440
h11v2        seed 1  beta [0.585, 0.472, 0.408, 0.436, 3.58]  asymmetry 0.190  first move 0.651  last move 1.3287
h11v2        seed 2  beta [0.548, 0.411, 0.32, 0.315, 3.213]  asymmetry 0.214  first move 0.482  last move 1.3263
h11v2-1step  seed 0  beta [4.194]  asymmetry 0.201  first move 1.547  last move 1.5467
h11v2-1step  seed 1  beta [4.642]  asymmetry 0.171  first move 1.569  last move 1.5695
h11v2-1step  seed 2  beta [4.184]  asymmetry 0.195  first move 1.551  last move 1.5510


## The Costs

Parameters; memory cells held per sequence (the softmax head's cache grows with the sequence, here 33 positions);
multiply-adds per token to write the memory (the delta rule's read-before-write doubles it) and per query to read it, counting the memory operations only (the
projections cost the same in every model, `2 * 64` inputs to each).

In [8]:
T_CTX = 2 * NPAIR + 1
costs = {}
for n in MODELS:
    torch.manual_seed(0)
    mod = Block(MODELS[n]())
    params = sum(p.numel() for p in mod.parameters() if p.requires_grad)
    mx = mod.mixer
    if isinstance(mx, HebbHopfield2):
        cells, write, read = mx.h * mx.d ** 2, 2 * mx.h * mx.d ** 2, mx.steps * mx.h * mx.d ** 2
    elif isinstance(mx, Delta):
        cells, write, read = mx.dk * mx.dv, 2 * mx.dk * mx.dv, mx.dk * mx.dv
    elif isinstance(mx, HebbHopfield):
        cells, write, read = mx.d ** 2, mx.d ** 2, mx.steps * mx.d ** 2
    elif isinstance(mx, Hebbian):
        cells, write, read = mx.dk * mx.dv, mx.dk * mx.dv, mx.dk * mx.dv
    elif isinstance(mx, Softmax):
        cells, write, read = T_CTX * (mx.dk + mx.dv), 0, T_CTX * (mx.dk + mx.dv)
    else:
        cells, write, read = 0, 0, 0
    costs[n] = (params, cells, write, read)
    cpu = np.mean([r[3] for r in results if r[0] == n])
    print(f"{n:14s} params {params:6d}  cells {cells:5d}  write/token {write:5d}  read/query {read:5d}  "
          f"training CPU {cpu:.0f} s")

delta-wide     params  63330  cells  1024  write/token  2048  read/query  1024  training CPU 157 s
h11v2          params  58561  cells  1024  write/token  2048  read/query  5120  training CPU 244 s
h11v2-1step    params  58557  cells  1024  write/token  2048  read/query  1024  training CPU 228 s


## Pretrained, Then Learning on Its Own

As in v1. Backpropagation pretrained only the slow weights, on 16 pairs; at
test every sequence brings bindings never seen, and the only thing that learns them is the memory each mixer writes as
the tokens arrive, with no gradient and no teacher. Here the trained models are frozen and given longer streams than
they were ever trained on - 8, 16, 24, 32 and 48 pairs, 10 000 sequences each from generator seed `999 + n` - to see
how the self-written memory copes with more to learn than it was shaped for. The softmax head's memory grows with the
stream; the Hebbian ones stay the same size.

In [9]:
LENGTHS = (8, 16, 24, 32, 48)
tests = {n: make_batch(torch.Generator().manual_seed(999 + n), TEST_N, n) for n in LENGTHS}
long_acc = {}
for name, seed, *_, state in results:
    torch.manual_seed(0)
    model = Block(MODELS[name]())
    model.load_state_dict(state)
    model.eval()
    with torch.no_grad():
        for n, (seq, y) in tests.items():
            long_acc.setdefault((name, n), []).append((model(seq).argmax(-1) == y).float().mean().item())
long_acc = {k: np.array(v) for k, v in long_acc.items()}
print("model          " + "".join(f"  {n:2d} pairs           " for n in LENGTHS))
for name in MODELS:
    row = ""
    for n in LENGTHS:
        m, lo, hi = interval(long_acc[(name, n)])
        row += f"  {m:.3f} [{lo:.3f}, {hi:.3f}]"
    print(f"{name:13s}{row}")
print()
for a, b in [("h11v2", "delta-wide"), ("h11v2", "h11v2-1step")]:
    row = ""
    for n in LENGTHS:
        m, lo, hi = interval(long_acc[(a, n)] - long_acc[(b, n)])
        row += f"  {m:+.3f} [{lo:+.3f}, {hi:+.3f}]"
    print(f"{a} - {b}:{row}")

model             8 pairs             16 pairs             24 pairs             32 pairs             48 pairs           
delta-wide     1.000 [0.999, 1.000]  0.996 [0.995, 0.998]  0.978 [0.966, 0.989]  0.935 [0.922, 0.949]  0.805 [0.761, 0.850]
h11v2          0.980 [0.949, 1.011]  0.885 [0.736, 1.034]  0.762 [0.530, 0.993]  0.658 [0.404, 0.912]  0.506 [0.251, 0.762]
h11v2-1step    0.972 [0.927, 1.016]  0.860 [0.704, 1.017]  0.728 [0.515, 0.940]  0.617 [0.407, 0.826]  0.465 [0.249, 0.680]

h11v2 - delta-wide:  -0.020 [-0.050, +0.011]  -0.112 [-0.259, +0.036]  -0.216 [-0.437, +0.004]  -0.277 [-0.518, -0.036]  -0.299 [-0.511, -0.087]
h11v2 - h11v2-1step:  +0.009 [-0.028, +0.045]  +0.024 [-0.086, +0.135]  +0.034 [-0.087, +0.155]  +0.042 [-0.083, +0.167]  +0.042 [-0.081, +0.165]


## The Figures

The first: each seed's test accuracy at 16 pairs (dots), the mean and its 95% interval (bars), and chance (dashed
line). The second: accuracy against the length of the stream, the mean and its 95% interval over seeds, with the
trained length dotted.

In [10]:
plt.rcParams["svg.hashsalt"] = "h11"
os.makedirs("../docs/figures", exist_ok=True)
names = list(MODELS)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for i, n in enumerate(names):
    m, lo, hi = interval(acc[n])
    ax.errorbar([i], [m], yerr=[[m - lo], [hi - m]], fmt="s", color="black", capsize=4, ms=5)
    ax.scatter(i + np.linspace(-0.15, 0.15, len(acc[n])), acc[n], s=12, color="tab:blue", zorder=3)
ax.axhline(1 / NV, ls="--", color="grey", lw=1)
ax.set_xticks(range(len(names)), names, rotation=25, ha="right")
ax.set_ylabel("test recall accuracy")
ax.set_ylim(0, 1)
ax.set_title("Associative recall, 16 pairs, 3 seeds (pilot)")
fig.tight_layout()
fig.savefig("../docs/figures/h11v2_recall.svg", metadata={"Date": None})
plt.close(fig)
print("wrote docs/figures/h11v2_recall.svg")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for name, style in [("delta-wide", "^-"), ("h11v2", "s-"), ("h11v2-1step", "d--")]:
    ms, los, his = zip(*(interval(long_acc[(name, n)]) for n in LENGTHS))
    ax.errorbar(LENGTHS, ms, yerr=[np.subtract(ms, los), np.subtract(his, ms)], fmt=style, capsize=3, ms=4, label=name)
ax.axvline(NPAIR, ls=":", color="grey", lw=1)
ax.axhline(1 / NV, ls="--", color="grey", lw=1)
ax.set_xlabel("pairs in the stream (trained on 16)")
ax.set_ylabel("test recall accuracy")
ax.set_ylim(0, 1)
ax.set_title("Frozen weights, memory written on its own")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("../docs/figures/h11v2_lengths.svg", metadata={"Date": None})
plt.close(fig)
print("wrote docs/figures/h11v2_lengths.svg")

wrote docs/figures/h11v2_recall.svg


wrote docs/figures/h11v2_lengths.svg
